<a href="https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Scope.** This notebook documents the feature design and leakage checks used in the capstone (`ai_opportunity` lane, 10,550 pages, 29 clients). It needs no gated data: the code below builds the features on a tiny **toy table (invented numbers, only to show the definitions)** and checks the feature catalogue. The real run on the lane data is in `work/notebooks/capstone_01_decline_prediction.ipynb`. Paper: https://najam2005.github.io/flyrank-ml-internship/paper/

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The label is *declining* if last-30-day impressions fall below 0.8 x previous-30-day impressions. Every feature must come from time **before** that label window, so no 90-day or last-30-day column is used directly. The oldest 30-day window is derived as the 90-day total minus the two later windows (this assumes three exact, non-overlapping periods).

Features built: log previous-window impressions, log oldest-window impressions, their difference (momentum), a no-old-traffic flag, a page-age tier, and three static descriptors (search intent, word-count tier, character-count tier). Categories keep an explicit `unknown` level so a missing descriptor never silently becomes a number.

In [1]:
import numpy as np
import pandas as pd

# TOY rows (invented) with the same column roles as the lane data. Not real data.
toy = pd.DataFrame({
    "impr_90d":       [2100, 900, 640, 3000, 150],
    "impr_prev_30d":  [1000, 500, 300, 1500, 120],
    "impr_last_30d":  [700, 480, 340, 1400, 30],
    "page_age_days":  [200, 120, 400, 60, 45],
    "intent":         ["informational", "commercial", None, "informational", "navigational"],
    "words_tier":     ["1000-2000", "<1000", "2000-3500", None, "<1000"],
    "chars_tier":     ["mid", "low", "high", "mid", None],
})

AGE_BINS   = [30, 90, 180, 365, np.inf]
AGE_LABELS = ["31-90", "91-180", "181-365", "365+"]

def build_features(df):
    out = pd.DataFrame(index=df.index)
    oldest = df["impr_90d"] - df["impr_prev_30d"] - df["impr_last_30d"]   # derived oldest 30-day window
    out["log_prev"]    = np.log1p(df["impr_prev_30d"])
    out["log_oldest"]  = np.log1p(oldest.clip(lower=0))
    out["momentum"]    = out["log_prev"] - out["log_oldest"]
    out["no_old_flag"] = (oldest <= 0).astype(int)
    out["age_tier"]    = pd.cut(df["page_age_days"], AGE_BINS, labels=AGE_LABELS).astype("object").fillna("unknown")
    for col in ["intent", "words_tier", "chars_tier"]:
        out[col] = df[col].fillna("unknown")
    return out

features = build_features(toy)
display(features)
print("Feature columns:", list(features.columns))

,log_prev,log_oldest,momentum,no_old_flag,age_tier,intent,words_tier,chars_tier
0,6.908755,5.993961,0.914793,0,181-365,informational,1000-2000,mid
1,6.216606,0.000000,6.216606,1,91-180,commercial,<1000,low
2,5.707110,0.000000,5.707110,1,365+,unknown,2000-3500,high
3,7.313887,4.615121,2.698766,0,31-90,informational,unknown,mid
4,4.795791,0.000000,4.795791,1,31-90,navigational,<1000,unknown


Feature columns: ['log_prev', 'log_oldest', 'momentum', 'no_old_flag', 'age_tier', 'intent', 'words_tier', 'chars_tier']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

All features are available **before** the label window starts. Oldest-window traffic is the only derived one. Pages with no traffic in the oldest window are marked by `no_old_flag`; this is mostly a page-age effect (in an earlier development check, 57.8% of pages aged 31-90 days had no old-window traffic vs at most 0.2% in older tiers). Page age, word count and character count are coarse tiers and may be measured at the end of the label window, so their timing is a stated limitation.

In [2]:
notes = pd.DataFrame([
    ["log_prev",    "log of impressions in the previous 30 days",              "no missing (pages need >= 100 previous impressions)", "numeric",     "yes: window ends before the label window"],
    ["log_oldest",  "log of impressions in the oldest 30 days (derived)",      "negative derived values clipped to 0",                "numeric",     "yes: window is older than both later windows"],
    ["momentum",    "log_prev minus log_oldest (rising or falling demand)",    "follows from the two inputs",                         "numeric",     "yes"],
    ["no_old_flag", "1 if the page had no traffic in the oldest window",       "no missing",                                          "binary",      "yes (mostly a young-page marker)"],
    ["age_tier",    "page age band: 31-90, 91-180, 181-365, 365+ days",        "'unknown' level",                                     "categorical", "mostly; coarse tier, may be measured late"],
    ["intent",      "search intent of the page",                               "'unknown' level",                                     "categorical", "static descriptor"],
    ["words_tier",  "word-count band",                                         "'unknown' level",                                     "categorical", "static, but may be measured late"],
    ["chars_tier",  "character-count band",                                    "'unknown' level",                                     "categorical", "static, but may be measured late"],
], columns=["feature", "meaning", "missing_handling", "type", "available_before_prediction"])
pd.set_option("display.max_colwidth", None)
display(notes)

,feature,meaning,missing_handling,type,available_before_prediction
0,log_prev,log of impressions in the previous 30 days,no missing (pages need >= 100 previous impressions),numeric,yes: window ends before the label window
1,log_oldest,log of impressions in the oldest 30 days (derived),negative derived values clipped to 0,numeric,yes: window is older than both later windows
2,momentum,log_prev minus log_oldest (rising or falling demand),follows from the two inputs,numeric,yes
3,no_old_flag,1 if the page had no traffic in the oldest window,no missing,binary,yes (mostly a young-page marker)
4,age_tier,"page age band: 31-90, 91-180, 181-365, 365+ days",'unknown' level,categorical,"mostly; coarse tier, may be measured late"
5,intent,search intent of the page,'unknown' level,categorical,static descriptor
6,words_tier,word-count band,'unknown' level,categorical,"static, but may be measured late"
7,chars_tier,character-count band,'unknown' level,categorical,"static, but may be measured late"


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Rule: a feature is leaky if it contains the label window (the last 30 days) or is derived from the label. Three tests were used:

1. **Window test.** Every 90-day and last-30-day column contains the label window, so none is a feature.
2. **Lane-join test (measured).** `engagement_fix` shares 10,677 pages and all 33 clients with this lane, but its 30-day traffic columns equal this lane's *last* window in 100% of matched pages, so they would leak the label and were not used. Only its static descriptors were taken.
3. **Perfect-separation test.** Any single feature that separates the label almost perfectly (AUC near 1) is suspicious. The cell below shows the test on the toy table: the leaky column is caught, the real features are not.

Two honest limits: the oldest-window feature relies on a window-alignment assumption, and lane membership in `engagement_fix` may depend on recent performance, which is why 858 pages outside it were excluded from the main analysis.

In [3]:
def auc_rank(score, y):
    """AUC by ranks (no sklearn needed). 0.5 = random, 1.0 = perfect."""
    score, y = pd.Series(score).reset_index(drop=True), pd.Series(y).reset_index(drop=True)
    ranks = score.rank()
    n1, n0 = int((y == 1).sum()), int((y == 0).sum())
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

# Toy label, same rule as the capstone: last_30d < 0.8 x prev_30d
y = (toy["impr_last_30d"] < 0.8 * toy["impr_prev_30d"]).astype(int)

candidates = features.select_dtypes("number").copy()
candidates["LEAKY_last_30d_ratio"] = toy["impr_last_30d"] / toy["impr_prev_30d"]   # contains the label window

rows = []
for col in candidates.columns:
    a = auc_rank(candidates[col], y)
    best = max(a, 1 - a)                      # direction does not matter
    rows.append([col, round(a, 3), "LEAK SUSPECT" if best >= 0.99 else "ok"])
display(pd.DataFrame(rows, columns=["column", "auc_vs_label", "verdict"]))
print("Toy label rate:", y.mean())
print("Toy table is tiny (5 rows): this only shows how the test works, not a real result.")

,column,auc_vs_label,verdict
0,log_prev,0.333,ok
1,log_oldest,0.667,ok
2,momentum,0.167,ok
3,no_old_flag,0.417,ok
4,LEAKY_last_30d_ratio,0.000,LEAK SUSPECT


Toy label rate: 0.4
Toy table is tiny (5 rows): this only shows how the test works, not a real result.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded on purpose, each for a stated reason. No client names, page identifiers, URLs or queries are used anywhere; hashed identifiers are used only for grouping clients in validation.

In [4]:
excluded = pd.DataFrame([
    ["every 90-day column",                        "contains the label window (last 30 days)"],
    ["every last-30-day column",                   "is the label window"],
    ["trend_* columns",                            "computed from the label window"],
    ["health_score",                               "derived from recent performance (label-adjacent)"],
    ["is_declining",                               "is the label itself"],
    ["product flags",                              "set by FlyRank rules that may use recent performance"],
    ["engagement_fix traffic columns",             "equal this lane's last window in 100% of matched pages (measured)"],
    ["hashed identifiers",                         "used only to group by client; not a feature"],
    ["clicks, CTR, position",                      "cannot be used without leaking the label window"],
    ["freshness tier",                             "measured at the end of the label window; sensitivity run only"],
    ["AI-referral columns",                        "empty in this lane"],
    ["client names, domains, URLs, queries",       "privacy: never used or published"],
], columns=["excluded_field", "why"])
display(excluded)
print(len(excluded), "exclusions listed")

,excluded_field,why
0,every 90-day column,contains the label window (last 30 days)
1,every last-30-day column,is the label window
2,trend_* columns,computed from the label window
3,health_score,derived from recent performance (label-adjacent)
4,is_declining,is the label itself
5,product flags,set by FlyRank rules that may use recent performance
6,engagement_fix traffic columns,equal this lane's last window in 100% of matched pages (measured)
7,hashed identifiers,used only to group by client; not a feature
8,"clicks, CTR, position",cannot be used without leaking the label window
9,freshness tier,measured at the end of the label window; sensitivity run only


12 exclusions listed


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.